# EcoSort: add RealWaste and compare models
Use a T4 GPU and run the code cell. Existing Colab inputs are reused where available; otherwise upload the requested original TrashNet split ZIP and previous TACO result ZIP. RealWaste downloads automatically (~657 MB). This trains a separate ResNet18 for up to 15 epochs, selects it using validation, and compares both models on the same test sets.

Source: [RealWaste, UCI](https://archive.ics.uci.edu/dataset/908/realwaste), Single, Iranmanesh and Raad (2023), CC BY 4.0. Only the six compatible categories are used. This notebook has not been GPU-trained merely by opening it. Keep your previous result ZIP.

In [ ]:
# Expanded data experiment: existing classes + TACO + RealWaste.
from pathlib import Path
import json, sys, shutil, zipfile, hashlib, traceback, importlib.metadata
import torch
from google.colab import files
from IPython.display import display, Image as DisplayImage
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before running.')
work=Path('/content/ecosort_realwaste_work');work.mkdir(parents=True,exist_ok=True)
source=work/'Src';source.mkdir(exist_ok=True)
MODULES={'taco_data': '"""TACO plastic-packaging crops, with source-grouped splits and provenance."""\nimport concurrent.futures\nimport hashlib\nimport io\nimport json\nimport math\nimport random\nimport urllib.request\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom PIL import Image, ImageDraw, ImageOps\n\nTACO_COMMIT = \'70479ad9adc3da971a8ae75ac97eb1026910eec7\'\nANNOTATIONS_URL = f\'https://raw.githubusercontent.com/pedropro/TACO/{TACO_COMMIT}/data/annotations.json\'\nCATEGORIES = {\'Plastic film\', \'Other plastic wrapper\', \'Crisp packet\'}\nCLASSES = [\'cardboard\', \'glass\', \'metal\', \'paper\', \'plastic\', \'trash\']\nPOLICY = (\'EcoSort plastic includes visually identified flexible plastic packaging, including \'\n          \'metallized snack packets, following the project packaging convention. \'\n          \'Source labels do not verify chemical composition or recyclability.\')\n\ndef sha256(data):\n    return hashlib.sha256(data).hexdigest()\n\ndef pixel_hash(image):\n    rgb = image.convert(\'RGB\')\n    return sha256(str(rgb.size).encode() + rgb.tobytes())\n\ndef dhash(image):\n    small = image.convert(\'L\').resize((9, 8), Image.Resampling.LANCZOS)\n    pixels = list(small.getdata())\n    result = 0\n    for y in range(8):\n        for x in range(8):\n            result = (result << 1) | (pixels[y*9+x] > pixels[y*9+x+1])\n    return result\n\ndef crop_box(bbox, original_size, downloaded_size):\n    x, y, width, height = bbox\n    sx, sy = downloaded_size[0]/original_size[0], downloaded_size[1]/original_size[1]\n    return (max(0, math.floor(x*sx)), max(0, math.floor(y*sy)),\n            min(downloaded_size[0], math.ceil((x+width)*sx)),\n            min(downloaded_size[1], math.ceil((y+height)*sy)))\n\ndef fetch_bytes(url, timeout=12):\n    request = urllib.request.Request(url, headers={\'User-Agent\': \'EcoSort-AI research dataset preparation\'})\n    with urllib.request.urlopen(request, timeout=timeout) as response:\n        return response.read()\n\nclass UnionFind:\n    def __init__(self, values):\n        self.parent = {v: v for v in values}\n    def find(self, value):\n        while self.parent[value] != value:\n            self.parent[value] = self.parent[self.parent[value]]\n            value = self.parent[value]\n        return value\n    def union(self, a, b):\n        a, b = self.find(a), self.find(b)\n        if a != b:\n            self.parent[max(a, b)] = min(a, b)\n\ndef grouped_splits(groups, seed=42):\n    """Input maps group IDs to crisp-packet presence; every group stays in one split."""\n    assignments = {}\n    for has_crisp in (False, True):\n        ids = sorted(k for k, flag in groups.items() if flag == has_crisp)\n        random.Random(seed + int(has_crisp)).shuffle(ids)\n        n = len(ids)\n        n_val = max(1, round(n*.15)) if n >= 7 else 0\n        n_test = n_val\n        for index, group_id in enumerate(ids):\n            assignments[group_id] = (\'val\' if index < n_val else\n                                      \'test\' if index < n_val+n_test else \'train\')\n    return assignments\n\ndef prepare_taco(output, annotation_bytes=None, workers=12, min_side=48):\n    """Download official preview images; keep only resolvable crops >= min_side pixels."""\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    raw = annotation_bytes if annotation_bytes is not None else fetch_bytes(ANNOTATIONS_URL)\n    annotations = json.loads(raw)\n    (output/\'annotations_source.json\').write_bytes(raw)\n    categories = {c[\'id\']: c[\'name\'] for c in annotations[\'categories\']}\n    images = {im[\'id\']: im for im in annotations[\'images\']}\n    selected = [a for a in annotations[\'annotations\'] if categories[a[\'category_id\']] in CATEGORIES]\n    by_image = defaultdict(list)\n    for ann in selected:\n        by_image[ann[\'image_id\']].append(ann)\n    cache = output/\'source_cache\'\n    cache.mkdir(exist_ok=True)\n    def download(image_id):\n        info = images[image_id]\n        path = cache/f\'image_{image_id:05d}.jpg\'\n        urls = list(dict.fromkeys(u for u in (info.get(\'flickr_640_url\'), info.get(\'flickr_url\')) if u))\n        if path.exists():\n            with Image.open(path) as im:\n                image = im.convert(\'RGB\')\n            if max(image.size) > 640:\n                image = ImageOps.contain(image, (640, 640), Image.Resampling.LANCZOS)\n                image.save(path, quality=95)\n                with Image.open(path) as im:\n                    image = im.convert(\'RGB\')\n            return image_id, image, (path.with_suffix(\'.url\').read_text() if path.with_suffix(\'.url\').exists() else \'cached URL unknown\'), None\n        failures = []\n        for url in urls:\n            for attempt in range(1):\n                try:\n                    content = fetch_bytes(url)\n                    with Image.open(io.BytesIO(content)) as im:\n                        image = ImageOps.contain(im.convert(\'RGB\'), (640, 640), Image.Resampling.LANCZOS)\n                    image.save(path, quality=95)\n                    path.with_suffix(\'.url\').write_text(url)\n                    # Return the same cached decoded pixels on initial and repeat runs.\n                    with Image.open(path) as im:\n                        return image_id, im.convert(\'RGB\'), url, None\n                except Exception as exc:\n                    failures.append(type(exc).__name__ + \': \' + str(exc))\n        return image_id, None, None, failures\n    downloaded = {}\n    errors = []\n    ids = sorted(by_image)\n    print(f\'TACO: {len(selected)} annotated candidate objects in {len(ids)} source images.\', flush=True)\n    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as pool:\n        for i, (image_id, image, url, error) in enumerate(pool.map(download, ids), 1):\n            if image is None:\n                errors.append({\'image_id\': image_id, \'errors\': error})\n            else:\n                downloaded[image_id] = (image, url)\n            if i % 25 == 0 or i == len(ids):\n                print(f\'Downloaded/checking {i}/{len(ids)} sources; failures {len(errors)}.\', flush=True)\n    # Group exact copies and near-identical whole scenes BEFORE deciding splits.\n    uf = UnionFind(sorted(downloaded))\n    fingerprints = {}\n    for image_id in sorted(downloaded):\n        image, _ = downloaded[image_id]\n        fingerprints[image_id] = (pixel_hash(image), dhash(image))\n    scene_ids = sorted(downloaded)\n    for i, a in enumerate(scene_ids):\n        for b in scene_ids[:i]:\n            if fingerprints[a][0] == fingerprints[b][0] or (fingerprints[a][1]^fingerprints[b][1]).bit_count() <= 4:\n                uf.union(a, b)\n    candidates = []\n    crop_fingerprints = {}\n    skipped = Counter()\n    for image_id in scene_ids:\n        image, url = downloaded[image_id]\n        info = images[image_id]\n        for ann in sorted(by_image[image_id], key=lambda x:x[\'id\']):\n            box = crop_box(ann[\'bbox\'], (info[\'width\'], info[\'height\']), image.size)\n            if box[2]-box[0] < min_side or box[3]-box[1] < min_side:\n                skipped[\'too_small\'] += 1\n                continue\n            crop = image.crop(box)\n            digest = pixel_hash(crop)\n            if digest in crop_fingerprints:\n                uf.union(image_id, crop_fingerprints[digest])\n                skipped[\'exact_crop_duplicate\'] += 1\n                continue\n            crop_fingerprints[digest] = image_id\n            candidates.append((image_id, ann, crop, digest, url))\n    group_flags = {}\n    for image_id, ann, crop, digest, url in candidates:\n        gid = uf.find(image_id)\n        group_flags[gid] = group_flags.get(gid, False) or categories[ann[\'category_id\']] == \'Crisp packet\'\n    split_map = grouped_splits(group_flags)\n    records = []\n    for image_id, ann, crop, digest, url in candidates:\n        split = split_map[uf.find(image_id)]\n        relative = f\'{split}/plastic/taco_{image_id:05d}_{ann["id"]:05d}.png\'\n        path = output/relative\n        path.parent.mkdir(parents=True, exist_ok=True)\n        crop.save(path)\n        records.append({\'file\':relative,\'image_id\':image_id,\'annotation_id\':ann[\'id\'],\n            \'group_id\':uf.find(image_id),\'split\':split,\'label\':\'plastic\',\n            \'source_category\':categories[ann[\'category_id\']], \'bbox\':ann[\'bbox\'],\n            \'downloaded_crop_size\':list(crop.size),\'pixel_sha256\':digest,\n            \'download_url\':url,\'original_url\':images[image_id].get(\'flickr_url\'),\n            \'license_id\':images[image_id].get(\'license\')})\n    counts = Counter(r[\'split\'] for r in records)\n    (output/\'download_errors.json\').write_text(json.dumps(errors,indent=2))\n    for split in (\'train\',\'val\',\'test\'):\n        if counts[split] < 15:\n            raise RuntimeError(f\'Too few usable TACO {split} crops ({counts[split]}). No training started. Review download_errors.json and source availability.\')\n    manifest = {\'taco_commit\':TACO_COMMIT,\'annotations_url\':ANNOTATIONS_URL,\n        \'annotations_sha256\':sha256(raw),\'candidate_objects\':len(selected),\n        \'downloaded_source_images\':len(downloaded),\'failed_source_images\':len(errors),\n        \'counts\':dict(counts),\'source_category_counts\':dict(Counter(r[\'source_category\'] for r in records)),\n        \'skip_counts\':dict(skipped),\'min_crop_side\':min_side,\'seed\':42,\n        \'split_unit\':\'source scene / exact crop duplicate / whole-image dHash distance <= 4\',\n        \'label_policy\':POLICY, \'licenses\':annotations.get(\'licenses\',[]),\'records\':records}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    (output/\'download_errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'Usable new TACO crops:\', dict(counts), \'Total:\', len(records), flush=True)\n    # Show only training examples; leave test images and scores untouched until final evaluation.\n    sample=random.Random(42).sample([r for r in records if r[\'split\']==\'train\'],min(30,counts[\'train\']))\n    sheet=Image.new(\'RGB\',(900,6*145),\'white\')\n    draw=ImageDraw.Draw(sheet)\n    for i,r in enumerate(sample):\n        with Image.open(output/r[\'file\']) as im:\n            preview=ImageOps.contain(im,(170,115))\n        x,y=(i%5)*180,(i//5)*145\n        sheet.paste(preview,(x,y))\n        draw.text((x,y+117),f\'{r["annotation_id"]} {r["source_category"][:19]}\',fill=\'black\')\n    sheet.save(output/\'training_preview.jpg\')\n    return manifest\n\nif __name__ == \'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--output\',required=True)\n    args=parser.parse_args()\n    prepare_taco(args.output)\n', 'train_taco': '"""Train a fresh six-class classifier with TACO plastic-packaging crops."""\nimport csv\nimport hashlib\nimport json\nimport random\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom PIL import Image\nfrom taco_data import CLASSES, POLICY, pixel_hash\n\n\ndef read_trashnet(archive, destination):\n    """Extract only class images. Never overwrite or modify the uploaded archive."""\n    destination = Path(destination)\n    records = []\n    with zipfile.ZipFile(archive) as z:\n        members = []\n        for member in z.infolist():\n            parts = Path(member.filename.replace(\'\\\\\', \'/\')).parts\n            for i in range(len(parts)-2):\n                if (parts[i] in (\'train\',\'val\',\'test\') and parts[i+1] in CLASSES\n                        and len(parts[i:]) == 3 and Path(parts[-1]).suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\')):\n                    members.append((member, parts[i], parts[i+1], parts[-1]))\n                    break\n        if not members:\n            raise ValueError(\'Upload EcoSort_TrashNet_split.zip containing train/val/test class folders.\')\n        seen_paths = set()\n        for member, split, label, name in members:\n            path = destination/split/label/name\n            if str(path) in seen_paths:\n                raise ValueError(f\'Duplicate ZIP destination: {split}/{label}/{name}\')\n            seen_paths.add(str(path))\n            path.parent.mkdir(parents=True,exist_ok=True)\n            path.write_bytes(z.read(member))\n            with Image.open(path) as im:\n                digest = pixel_hash(im)\n            records.append({\'file\':str(path),\'split\':split,\'label\':label,\'source\':\'trashnet\',\'pixel_sha256\':digest})\n    for split in (\'train\',\'val\',\'test\'):\n        labels = {r[\'label\'] for r in records if r[\'split\']==split}\n        if labels != set(CLASSES):\n            raise ValueError(f\'TrashNet {split} is missing classes: {set(CLASSES)-labels}\')\n    # Remove EVERY member of groups crossing a split or label, rather than choosing a label by guesswork.\n    by_hash = defaultdict(list)\n    for r in records:\n        by_hash[r[\'pixel_sha256\']].append(r)\n    keep, excluded = [], []\n    for group in by_hash.values():\n        conflict = len({(r[\'split\'],r[\'label\']) for r in group}) > 1\n        if conflict:\n            excluded.extend(dict(r,reason=\'duplicate crossing splits or labels\') for r in group)\n        else:\n            keep.append(group[0])\n            excluded.extend(dict(r,reason=\'duplicate within class and split\') for r in group[1:])\n    return keep, excluded\n\n\ndef prepare_records(archive, taco_root, work, data_root=None):\n    work = Path(work)\n    work.mkdir(parents=True,exist_ok=True)\n    original, excluded = read_trashnet(archive,Path(data_root) if data_root else work/\'trashnet\')\n    taco_root = Path(taco_root)\n    manifest = json.loads((taco_root/\'manifest.json\').read_text())\n    original_hashes = {r[\'pixel_sha256\'] for r in original}\n    blocked = {r[\'group_id\'] for r in manifest[\'records\'] if r[\'pixel_sha256\'] in original_hashes}\n    new = []\n    groups = defaultdict(set)\n    for r in manifest[\'records\']:\n        groups[r[\'group_id\']].add(r[\'split\'])\n        row = dict(r,file=str(taco_root/r[\'file\']),source=\'taco\')\n        if r[\'group_id\'] in blocked:\n            excluded.append(dict(row,reason=\'TACO group matches TrashNet decoded pixels\'))\n        else:\n            new.append(row)\n    if any(len(splits)>1 for splits in groups.values()):\n        raise ValueError(\'TACO source group crosses splits; training stopped.\')\n    counts = Counter(r[\'split\'] for r in new)\n    if counts[\'train\']<100 or counts[\'val\']<15 or counts[\'test\']<15:\n        raise ValueError(f\'Not enough NEW wrapper images to train: {dict(counts)}\')\n    all_records = original+new\n    split_hashes = defaultdict(set)\n    for r in all_records:\n        split_hashes[r[\'pixel_sha256\']].add(r[\'split\'])\n    if any(len(s)>1 for s in split_hashes.values()):\n        raise ValueError(\'Exact image leakage detected between splits.\')\n    (work/\'excluded_images.json\').write_text(json.dumps(excluded,indent=2))\n    (work/\'dataset_records.json\').write_text(json.dumps(all_records,indent=2))\n    with open(archive, \'rb\') as handle:\n        archive_digest = hashlib.file_digest(handle, \'sha256\').hexdigest()\n    summary = {\'archive_name\':Path(archive).name,\'archive_sha256\':archive_digest,\n        \'trashnet_counts\':dict(Counter(r[\'split\'] for r in original)),\n        \'new_taco_counts\':dict(counts),\'excluded_count\':len(excluded),\'classes\':CLASSES,\n        \'class_counts_by_split\':{s:dict(Counter(r[\'label\'] for r in all_records if r[\'split\']==s)) for s in (\'train\',\'val\',\'test\')}}\n    (work/\'dataset_summary.json\').write_text(json.dumps(summary,indent=2))\n    print(\'TrashNet after duplicate exclusion:\',dict(Counter(r[\'split\'] for r in original)),flush=True)\n    print(\'NEW wrapper images:\',dict(counts),flush=True)\n    return all_records\n\n\ndef train(records, output, head_epochs=3, finetune_epochs=7, batch_size=32, device_name="cuda"):\n    import numpy as np\n    import torch\n    from torch import nn\n    from torch.utils.data import Dataset, DataLoader\n    from torchvision import models, transforms\n    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report\n    output=Path(output)\n    output.mkdir(parents=True,exist_ok=True)\n    if device_name == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime in Colab before training. No training started.\')\n    random.seed(42)\n    np.random.seed(42)\n    torch.manual_seed(42)\n    torch.cuda.manual_seed_all(42)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    device=torch.device(device_name)\n    mean,std=[.485,.456,.406],[.229,.224,.225]\n    evaluation=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    augmentation=transforms.Compose([transforms.RandomResizedCrop(224,scale=(.75,1.)),transforms.RandomHorizontalFlip(),\n        transforms.RandomRotation(15),transforms.ColorJitter(.15,.15,.15,.05),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    class Images(Dataset):\n        def __init__(self, rows, transform): self.rows,self.transform=rows,transform\n        def __len__(self): return len(self.rows)\n        def __getitem__(self,index):\n            r=self.rows[index]\n            with Image.open(r[\'file\']) as im: image=self.transform(im.convert(\'RGB\'))\n            return image,CLASSES.index(r[\'label\'])\n    split_rows={s:[r for r in records if r[\'split\']==s] for s in (\'train\',\'val\',\'test\')}\n    generator=torch.Generator().manual_seed(42)\n    loaders={s:DataLoader(Images(rows,augmentation if s==\'train\' else evaluation),batch_size=batch_size,\n        shuffle=s==\'train\',num_workers=0,generator=generator if s==\'train\' else None) for s,rows in split_rows.items()}\n    print(\'Device:\',torch.cuda.get_device_name(0) if device_name==\'cuda\' else device_name,flush=True)\n    print(\'Starting a NEW ImageNet-initialized ResNet18. No previous EcoSort weights loaded.\',flush=True)\n    model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)\n    model.fc=nn.Linear(model.fc.in_features,len(CLASSES))\n    model.to(device)\n    counts=Counter(r[\'label\'] for r in split_rows[\'train\'])\n    weights=torch.tensor([len(split_rows[\'train\'])/(len(CLASSES)*counts[c]) for c in CLASSES],device=device)\n    loss_fn=nn.CrossEntropyLoss(weight=weights)\n    def predict(rows):\n        model.eval()\n        truth,prediction=[],[]\n        with torch.inference_mode():\n            for x,y in DataLoader(Images(rows,evaluation),batch_size=batch_size,num_workers=0):\n                p=model(x.to(device)).argmax(1).cpu().tolist()\n                prediction.extend(p)\n                truth.extend(y.tolist())\n        return truth,prediction\n    def metrics(rows):\n        y,p=predict(rows)\n        result={\'n\':len(y),\'accuracy\':accuracy_score(y,p),\n            \'confusion_matrix\':confusion_matrix(y,p,labels=list(range(6))).tolist()}\n        if {r[\'label\'] for r in rows}=={\'plastic\'}:\n            result[\'plastic_recall\']=result[\'accuracy\']\n            result[\'note\']=\'Positive-only wrapper set: this measures recall, not six-class accuracy or plastic precision.\'\n            result[\'by_source_category\']={c:{\'n\':sum(r.get(\'source_category\')==c for r in rows),\n                \'plastic_recall\':sum(pp==4 for r,pp in zip(rows,p) if r.get(\'source_category\')==c)/sum(r.get(\'source_category\')==c for r in rows)}\n                for c in sorted({r[\'source_category\'] for r in rows})}\n        else:\n            result[\'macro_f1\']=f1_score(y,p,labels=list(range(6)),average=\'macro\',zero_division=0)\n            result[\'class_report\']=classification_report(y,p,labels=list(range(6)),target_names=CLASSES,output_dict=True,zero_division=0)\n        return result\n    history=[]\n    best=(-1.,-1.)\n    checkpoint=output/\'ecosort_resnet18_taco_plastic.pth\'\n    epoch=0\n    for stage,n_epochs in [(\'head\',head_epochs),(\'layer4\',finetune_epochs)]:\n        if stage==\'layer4\':\n            model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n        for name,param in model.named_parameters():\n            param.requires_grad=name.startswith(\'fc.\') or (stage==\'layer4\' and name.startswith(\'layer4.\'))\n        groups=[{\'params\':model.fc.parameters(),\'lr\':1e-3 if stage==\'head\' else 1e-4}]\n        if stage==\'layer4\': groups.append({\'params\':model.layer4.parameters(),\'lr\':1e-5})\n        optimizer=torch.optim.AdamW(groups,weight_decay=1e-4)\n        for _ in range(n_epochs):\n            epoch+=1\n            # Frozen layers and their batch-normalization statistics remain frozen.\n            model.eval()\n            model.fc.train()\n            if stage==\'layer4\': model.layer4.train()\n            total_loss,correct,n=0.,0,0\n            for x,y in loaders[\'train\']:\n                x,y=x.to(device),y.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                logits=model(x)\n                loss=loss_fn(logits,y)\n                loss.backward()\n                optimizer.step()\n                total_loss+=loss.item()*len(y)\n                correct+=(logits.argmax(1)==y).sum().item()\n                n+=len(y)\n            val=metrics(split_rows[\'val\'])\n            wrapper_val=metrics([r for r in split_rows[\'val\'] if r[\'source\']==\'taco\'])\n            row={\'epoch\':epoch,\'stage\':stage,\'train_loss\':total_loss/n,\'train_accuracy\':correct/n,\n                \'val_accuracy\':val[\'accuracy\'],\'val_macro_f1\':val[\'macro_f1\'],\'wrapper_val_recall\':wrapper_val[\'plastic_recall\']}\n            history.append(row)\n            print(f"Epoch {epoch}/{head_epochs+finetune_epochs}: val F1 {val[\'macro_f1\']:.3f}, wrapper val recall {wrapper_val[\'plastic_recall\']:.1%}",flush=True)\n            score=(val[\'macro_f1\'],val[\'accuracy\'])\n            if score>best:\n                best=score\n                torch.save({\'model_state_dict\':model.state_dict(),\'classes\':CLASSES,\'class_names\':CLASSES,\n                    \'epoch\':epoch,\'val_macro_f1\':best[0],\'image_size\':224,\'mean\':mean,\'std\':std,\n                    \'label_policy\':POLICY,\'initialization\':\'ImageNet1K V1; no previous EcoSort checkpoint\'},checkpoint)\n    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n    final={\'classes\':CLASSES,\'label_policy\':POLICY,\'selection\':\'highest combined validation macro F1, accuracy breaks ties\',\n        \'selected_epoch\':torch.load(checkpoint,map_location=\'cpu\',weights_only=True)[\'epoch\'],\n        \'trashnet_test\':metrics([r for r in split_rows[\'test\'] if r[\'source\']==\'trashnet\']),\n        \'new_wrapper_test\':metrics([r for r in split_rows[\'test\'] if r[\'source\']==\'taco\']),\n        \'combined_test\':metrics(split_rows[\'test\']),\n        \'limitations\':[\'TrashNet test has been inspected in earlier experiments; interpret as diagnostic.\',\n            \'TACO holdout is grouped by source and near-identical scenes, not independently collected deployment photos.\',\n            \'Packaging categories are project labels, not verified material composition or recycling advice.\',\n            \'TACO crops differ from whole-image TrashNet photos; evaluate fresh user photos before deployment.\']}\n    (output/\'metrics.json\').write_text(json.dumps(final,indent=2))\n    with (output/\'training_history.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(history[0]));writer.writeheader();writer.writerows(history)\n    print(\'Selected epoch:\',final[\'selected_epoch\'],flush=True)\n    print(\'TrashNet test accuracy:\',f"{final[\'trashnet_test\'][\'accuracy\']:.1%}",flush=True)\n    print(\'NEW wrapper test recall:\',f"{final[\'new_wrapper_test\'][\'plastic_recall\']:.1%}",flush=True)\n    return final\n', 'realwaste_data': '"""Add the six compatible RealWaste categories without changing existing splits."""\nimport hashlib\nimport io\nimport json\nimport random\nimport re\nimport urllib.request\nimport zipfile\nfrom collections import Counter,defaultdict\nfrom pathlib import Path,PurePosixPath\nfrom PIL import Image,ImageOps,ImageDraw\nfrom taco_data import CLASSES,UnionFind,dhash,pixel_hash\n\nURL=\'https://archive.ics.uci.edu/static/public/908/realwaste.zip\'\nCITATION=\'Single, S., Iranmanesh, S., & Raad, R. (2023). RealWaste. UCI Machine Learning Repository. https://doi.org/10.24432/C5SS4G\'\nMAPPING={\'Cardboard\':\'cardboard\',\'Glass\':\'glass\',\'Metal\':\'metal\',\'Miscellaneous Trash\':\'trash\',\'Paper\':\'paper\',\'Plastic\':\'plastic\'}\n\n\ndef download_realwaste(path):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    if path.exists() and zipfile.is_zipfile(path): return path\n    part=path.with_suffix(\'.zip.part\')\n    print(\'Downloading RealWaste (~657 MB).\',flush=True)\n    request=urllib.request.Request(URL,headers={\'User-Agent\':\'EcoSort-AI research\'})\n    with urllib.request.urlopen(request,timeout=60) as src,part.open(\'wb\') as dst:\n        total=0;last=0\n        while chunk:=src.read(1024*1024):\n            dst.write(chunk);total+=len(chunk)\n            if total-last>=64*1024*1024:\n                print(f\'RealWaste downloaded: {total//(1024*1024)} MiB\',flush=True);last=total\n    if not zipfile.is_zipfile(part): raise ValueError(\'RealWaste download is not a valid ZIP.\')\n    part.replace(path)\n    return path\n\n\ndef split_groups(rows,uf,seed=2026):\n    grouped=defaultdict(list)\n    for i,r in enumerate(rows): grouped[uf.find(i)].append(r)\n    strata=defaultdict(list)\n    for group,members in grouped.items():\n        # Cross-label near-duplicate groups remain intact; use a deterministic majority stratum.\n        counts=Counter(r[\'label\'] for r in members)\n        dominant=sorted(counts,key=lambda c:(-counts[c],c))[0]\n        strata[dominant].append(group)\n    result={}\n    for label in CLASSES:\n        groups=sorted(strata[label]);random.Random(seed+CLASSES.index(label)).shuffle(groups)\n        if len(groups)<7: raise ValueError(f\'Only {len(groups)} source groups for {label}; inspect grouping before training.\')\n        holdout=max(1,round(.15*len(groups)))\n        for i,group in enumerate(groups): result[group]=\'val\' if i<holdout else \'test\' if i<2*holdout else \'train\'\n    return result\n\n\ndef prepare_realwaste(archive,existing_records,output,block_size=20,near_threshold=4):\n    """Group filename-adjacent blocks and near copies before generating any split."""\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    cache=output/\'source_cache\';cache.mkdir(exist_ok=True)\n    existing_hashes={r[\'pixel_sha256\'] for r in existing_records}\n    candidates=[];skipped=[]\n    with zipfile.ZipFile(archive) as z:\n        members=[n for n in z.namelist() if PurePosixPath(n).parent.name in MAPPING and PurePosixPath(n).suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\')]\n        for i,name in enumerate(sorted(members)):\n            label=MAPPING[PurePosixPath(name).parent.name]\n            content=z.read(name)\n            with Image.open(io.BytesIO(content)) as im:\n                rgb=im.convert(\'RGB\');digest=pixel_hash(rgb);fingerprint=dhash(rgb)\n                if min(rgb.size)<96:\n                    skipped.append({\'archive_member\':name,\'reason\':\'image side below 96 pixels\'});continue\n            path=cache/f\'{label}_{PurePosixPath(name).name}\'\n            path.write_bytes(content)\n            number=re.search(r\'(\\d+)(?=\\.[^.]+$)\',name)\n            if not number: raise ValueError(f\'Cannot determine image sequence number: {name}\')\n            candidates.append({\'file\':str(path),\'label\':label,\'source\':\'realwaste\',\'archive_member\':name,\n                \'pixel_sha256\':digest,\'dhash\':fingerprint,\'sequence_block\':(int(number.group())-1)//block_size})\n            if (i+1)%500==0: print(f\'RealWaste inspected {i+1}/{len(members)} images\',flush=True)\n    uf=UnionFind(range(len(candidates)))\n    blocks={};seen={};conflicting=set();duplicates=set();collisions=set()\n    for i,r in enumerate(candidates):\n        key=(r[\'label\'],r[\'sequence_block\'])\n        if key in blocks: uf.union(i,blocks[key])\n        else: blocks[key]=i\n        digest=r[\'pixel_sha256\']\n        if digest in existing_hashes: collisions.add(i)\n        if digest in seen:\n            j=seen[digest];uf.union(i,j)\n            if r[\'label\']!=candidates[j][\'label\']: conflicting.update([i,j])\n            else: duplicates.add(i)\n        else: seen[digest]=i\n        for j,previous in enumerate(candidates[:i]):\n            if (r[\'dhash\']^previous[\'dhash\']).bit_count()<=near_threshold: uf.union(i,j)\n    # A source-group collision with an existing experiment excludes the whole new group.\n    blocked={uf.find(i) for i in collisions|conflicting}\n    assignments=split_groups(candidates,uf)\n    records=[]\n    for i,r in enumerate(candidates):\n        group=uf.find(i)\n        reason=(\'group contains existing-dataset pixels or conflicting exact labels\' if group in blocked else\n                \'exact duplicate within the same class\' if i in duplicates else None)\n        if reason: skipped.append(dict(r,reason=reason));continue\n        records.append(dict(r,group_id=f\'realwaste_{group}\',split=assignments[group]))\n    counts={s:dict(Counter(r[\'label\'] for r in records if r[\'split\']==s)) for s in (\'train\',\'val\',\'test\')}\n    for s in counts:\n        for c in CLASSES:\n            if counts[s].get(c,0)<(30 if s==\'train\' else 10):\n                raise ValueError(f\'Insufficient RealWaste {s}/{c} after grouping/exclusion: {counts[s].get(c,0)}\')\n    digest=hashlib.file_digest(open(archive,\'rb\'),\'sha256\').hexdigest()\n    manifest={\'source_url\':URL,\'source_doi\':\'https://doi.org/10.24432/C5SS4G\',\'citation\':CITATION,\n        \'license\':\'CC BY 4.0\',\'archive_sha256\':digest,\'class_mapping\':MAPPING,\n        \'unmapped_classes_excluded\':[\'Food Organics\',\'Textile Trash\',\'Vegetation\'],\n        \'candidate_images\':len(members),\'usable_images\':len(records),\'counts\':counts,\n        \'seed\':2026,\'sequence_block_size\':block_size,\'near_duplicate_dhash_threshold\':near_threshold,\n        \'split_policy\':\'Approximately 70/15/15 by grouped source blocks, stratified on majority class. Groups combine within-class filename blocks and near-identical images.\',\n        \'limitation\':\'Filename blocks are a conservative sequence heuristic, not verified capture sessions. Perceptual hashing cannot find every related scene. RealWaste comes from one landfill environment.\',\n        \'records\':records,\'excluded\':skipped}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    validate_splits(existing_records+records)\n    preview(records,output/\'training_preview.jpg\')\n    print(\'NEW RealWaste usable images:\',len(records),flush=True)\n    print(\'NEW RealWaste split totals:\',dict(Counter(r[\'split\'] for r in records)),flush=True)\n    return manifest\n\n\ndef validate_splits(records):\n    pixels=defaultdict(set);groups=defaultdict(set)\n    for r in records:\n        pixels[r[\'pixel_sha256\']].add(r[\'split\'])\n        if \'group_id\' in r: groups[(r[\'source\'],r[\'group_id\'])].add(r[\'split\'])\n    if any(len(v)>1 for v in pixels.values()): raise ValueError(\'Exact pixels cross splits.\')\n    if any(len(v)>1 for v in groups.values()): raise ValueError(\'Source groups cross splits.\')\n\n\ndef preview(records,path):\n    sheet=Image.new(\'RGB\',(1000,840),\'white\');draw=ImageDraw.Draw(sheet)\n    for ci,label in enumerate(CLASSES):\n        pool=[r for r in records if r[\'split\']==\'train\' and r[\'label\']==label]\n        for j,r in enumerate(random.Random(42).sample(pool,min(5,len(pool)))):\n            with Image.open(r[\'file\']) as im: image=ImageOps.contain(im.convert(\'RGB\'),(190,110))\n            x,y=j*200,ci*140;sheet.paste(image,(x,y));draw.text((x,y+115),label,fill=\'black\')\n    sheet.save(path)\n', 'train_multisource': '"""Expanded-data ResNet18 training with early stopping and same-set baseline comparison."""\nimport csv\nimport json\nimport random\nfrom collections import Counter\nfrom pathlib import Path\nfrom PIL import Image\nfrom taco_data import CLASSES, POLICY\n\n\ndef train(records, output, head_epochs=3, finetune_epochs=7, batch_size=32, device_name="cuda",\n          checkpoint_name="ecosort_resnet18_taco_plastic.pth", baseline_checkpoint=None, patience=None):\n    import numpy as np\n    import torch\n    from torch import nn\n    from torch.utils.data import Dataset, DataLoader\n    from torchvision import models, transforms\n    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report\n    output=Path(output)\n    output.mkdir(parents=True,exist_ok=True)\n    if device_name == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime in Colab before training. No training started.\')\n    random.seed(42)\n    np.random.seed(42)\n    torch.manual_seed(42)\n    torch.cuda.manual_seed_all(42)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    device=torch.device(device_name)\n    mean,std=[.485,.456,.406],[.229,.224,.225]\n    evaluation=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    augmentation=transforms.Compose([transforms.RandomResizedCrop(224,scale=(.75,1.)),transforms.RandomHorizontalFlip(),\n        transforms.RandomRotation(15),transforms.ColorJitter(.15,.15,.15,.05),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    class Images(Dataset):\n        def __init__(self, rows, transform): self.rows,self.transform=rows,transform\n        def __len__(self): return len(self.rows)\n        def __getitem__(self,index):\n            r=self.rows[index]\n            with Image.open(r[\'file\']) as im: image=self.transform(im.convert(\'RGB\'))\n            return image,CLASSES.index(r[\'label\'])\n    split_rows={s:[r for r in records if r[\'split\']==s] for s in (\'train\',\'val\',\'test\')}\n    generator=torch.Generator().manual_seed(42)\n    loaders={s:DataLoader(Images(rows,augmentation if s==\'train\' else evaluation),batch_size=batch_size,\n        shuffle=s==\'train\',num_workers=0,generator=generator if s==\'train\' else None) for s,rows in split_rows.items()}\n    print(\'Device:\',torch.cuda.get_device_name(0) if device_name==\'cuda\' else device_name,flush=True)\n    print(\'Starting a NEW ImageNet-initialized ResNet18. No previous EcoSort weights loaded.\',flush=True)\n    model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)\n    model.fc=nn.Linear(model.fc.in_features,len(CLASSES))\n    model.to(device)\n    counts=Counter(r[\'label\'] for r in split_rows[\'train\'])\n    weights=torch.tensor([len(split_rows[\'train\'])/(len(CLASSES)*counts[c]) for c in CLASSES],device=device)\n    loss_fn=nn.CrossEntropyLoss(weight=weights)\n    def predict(rows):\n        model.eval()\n        truth,prediction=[],[]\n        with torch.inference_mode():\n            for x,y in DataLoader(Images(rows,evaluation),batch_size=batch_size,num_workers=0):\n                p=model(x.to(device)).argmax(1).cpu().tolist()\n                prediction.extend(p)\n                truth.extend(y.tolist())\n        return truth,prediction\n    def metrics(rows):\n        y,p=predict(rows)\n        result={\'n\':len(y),\'accuracy\':accuracy_score(y,p),\n            \'confusion_matrix\':confusion_matrix(y,p,labels=list(range(6))).tolist()}\n        if {r[\'label\'] for r in rows}=={\'plastic\'}:\n            result[\'plastic_recall\']=result[\'accuracy\']\n            result[\'note\']=\'Positive-only wrapper set: this measures recall, not six-class accuracy or plastic precision.\'\n            result[\'by_source_category\']={c:{\'n\':sum(r.get(\'source_category\')==c for r in rows),\n                \'plastic_recall\':sum(pp==4 for r,pp in zip(rows,p) if r.get(\'source_category\')==c)/sum(r.get(\'source_category\')==c for r in rows)}\n                for c in sorted({r[\'source_category\'] for r in rows})}\n        else:\n            result[\'macro_f1\']=f1_score(y,p,labels=list(range(6)),average=\'macro\',zero_division=0)\n            result[\'class_report\']=classification_report(y,p,labels=list(range(6)),target_names=CLASSES,output_dict=True,zero_division=0)\n        return result\n    history=[]\n    best=(-1.,-1.)\n    checkpoint=output/checkpoint_name\n    best_epoch=0\n    epoch=0\n    for stage,n_epochs in [(\'head\',head_epochs),(\'layer4\',finetune_epochs)]:\n        if stage==\'layer4\':\n            model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n        for name,param in model.named_parameters():\n            param.requires_grad=name.startswith(\'fc.\') or (stage==\'layer4\' and name.startswith(\'layer4.\'))\n        groups=[{\'params\':model.fc.parameters(),\'lr\':1e-3 if stage==\'head\' else 1e-4}]\n        if stage==\'layer4\': groups.append({\'params\':model.layer4.parameters(),\'lr\':1e-5})\n        optimizer=torch.optim.AdamW(groups,weight_decay=1e-4)\n        for _ in range(n_epochs):\n            epoch+=1\n            # Frozen layers and their batch-normalization statistics remain frozen.\n            model.eval()\n            model.fc.train()\n            if stage==\'layer4\': model.layer4.train()\n            total_loss,correct,n=0.,0,0\n            for x,y in loaders[\'train\']:\n                x,y=x.to(device),y.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                logits=model(x)\n                loss=loss_fn(logits,y)\n                loss.backward()\n                optimizer.step()\n                total_loss+=loss.item()*len(y)\n                correct+=(logits.argmax(1)==y).sum().item()\n                n+=len(y)\n            val=metrics(split_rows[\'val\'])\n            wrapper_val=metrics([r for r in split_rows[\'val\'] if r[\'source\']==\'taco\'])\n            row={\'epoch\':epoch,\'stage\':stage,\'train_loss\':total_loss/n,\'train_accuracy\':correct/n,\n                \'val_accuracy\':val[\'accuracy\'],\'val_macro_f1\':val[\'macro_f1\'],\'wrapper_val_recall\':wrapper_val[\'plastic_recall\']}\n            history.append(row)\n            print(f"Epoch {epoch}/{head_epochs+finetune_epochs}: val F1 {val[\'macro_f1\']:.3f}, wrapper val recall {wrapper_val[\'plastic_recall\']:.1%}",flush=True)\n            score=(val[\'macro_f1\'],val[\'accuracy\'])\n            if score>best:\n                best=score\n                best_epoch=epoch\n                torch.save({\'model_state_dict\':model.state_dict(),\'classes\':CLASSES,\'class_names\':CLASSES,\n                    \'epoch\':epoch,\'val_macro_f1\':best[0],\'image_size\':224,\'mean\':mean,\'std\':std,\n                    \'label_policy\':POLICY,\'initialization\':\'ImageNet1K V1; no previous EcoSort checkpoint\'},checkpoint)\n            if stage==\'layer4\' and patience is not None and epoch-best_epoch>=patience:\n                print(\'Early stopping: validation selection score has not improved.\',flush=True)\n                break\n    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n    source_rows={s:[r for r in split_rows[\'test\'] if r[\'source\']==s] for s in sorted({r[\'source\'] for r in split_rows[\'test\']})}\n    source_test={s:metrics(rows) for s,rows in source_rows.items()}\n    baseline_test=None\n    if baseline_checkpoint is not None:\n        base=torch.load(baseline_checkpoint,map_location=device,weights_only=True)\n        if base[\'class_names\']!=CLASSES or base.get(\'mean\')!=mean or base.get(\'std\')!=std or base.get(\'image_size\')!=224:\n            raise ValueError(\'Baseline class order or preprocessing differs; comparison stopped.\')\n        model.load_state_dict(base[\'model_state_dict\'])\n        baseline_test={s:metrics(rows) for s,rows in source_rows.items()}\n        model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n    final={\'classes\':CLASSES,\'label_policy\':POLICY,\'selection\':\'highest combined validation macro F1, accuracy breaks ties\',\n        \'selected_epoch\':torch.load(checkpoint,map_location=\'cpu\',weights_only=True)[\'epoch\'],\n        \'trashnet_test\':source_test[\'trashnet\'],\n        \'new_wrapper_test\':source_test[\'taco\'],\n        \'source_test\':source_test,\'baseline_source_test\':baseline_test,\n        \'combined_test\':metrics(split_rows[\'test\']),\n        \'limitations\':[\'TrashNet test has been inspected in earlier experiments; interpret as diagnostic.\',\n            \'TACO holdout is grouped by source and near-identical scenes, not independently collected deployment photos.\',\n            \'Packaging categories are project labels, not verified material composition or recycling advice.\',\n            \'TACO crops differ from whole-image TrashNet photos; evaluate fresh user photos before deployment.\']}\n    (output/\'metrics.json\').write_text(json.dumps(final,indent=2))\n    with (output/\'training_history.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(history[0]));writer.writeheader();writer.writerows(history)\n    print(\'Selected epoch:\',final[\'selected_epoch\'],flush=True)\n    print(\'TrashNet test accuracy:\',f"{final[\'trashnet_test\'][\'accuracy\']:.1%}",flush=True)\n    print(\'NEW wrapper test recall:\',f"{final[\'new_wrapper_test\'][\'plastic_recall\']:.1%}",flush=True)\n    if \'realwaste\' in source_test:\n        print(\'NEW RealWaste test accuracy:\',f"{source_test[\'realwaste\'][\'accuracy\']:.1%}",flush=True)\n        if baseline_test is not None:\n            print(\'Previous model on SAME RealWaste test:\',f"{baseline_test[\'realwaste\'][\'accuracy\']:.1%}",flush=True)\n    return final\n'}
for name,text in MODULES.items(): (source/f'{name}.py').write_text(text)
sys.path.insert(0,str(source))
for name in MODULES: sys.modules.pop(name,None)
from taco_data import prepare_taco
from train_taco import prepare_records
from realwaste_data import download_realwaste, prepare_realwaste, validate_splits
from train_multisource import train

def locate_inputs():
    archive=Path('/content/EcoSort_TrashNet_split.zip')
    baseline=Path('/content/ecosort_taco_work/Result/Models/ecosort_resnet18_taco_plastic.pth')
    if not archive.exists(): archive=None
    if not baseline.exists(): baseline=None
    for path in sorted(Path('/content').glob('*.zip')):
        try:
            with zipfile.ZipFile(path) as z:
                names=z.namelist()
                if baseline is None and 'Models/ecosort_resnet18_taco_plastic.pth' in names:
                    baseline=work/'previous_taco_checkpoint.pth'
                    baseline.write_bytes(z.read('Models/ecosort_resnet18_taco_plastic.pth'))
                if archive is None and any('/train/cardboard/' in '/'+n.replace('\\','/') for n in names):
                    archive=path
        except zipfile.BadZipFile:
            continue
    return archive,baseline

archive,baseline=locate_inputs()
if archive is None or baseline is None:
    print('Upload the missing file(s):',flush=True)
    if archive is None: print('  EcoSort_TrashNet_split.zip')
    if baseline is None: print('  EcoSort_TACO_Plastic_Result.zip (previous trained result bundle)')
    print('You can select both ZIPs together. No Python environment folders are needed.')
    uploaded=files.upload();del uploaded
    archive,baseline=locate_inputs()
if archive is None or baseline is None:
    raise ValueError('Required TrashNet split ZIP or previous TACO result checkpoint was not found.')
print('TrashNet archive:',archive.name)
print('Previous checkpoint:',baseline.name)
old=torch.load(baseline,map_location='cpu',weights_only=True)
if old.get('class_names')!=['cardboard','glass','metal','paper','plastic','trash']:
    raise ValueError('Baseline has a different class order.')
del old
result=work/'Result';result.mkdir(exist_ok=True)
(result/'Src').mkdir(exist_ok=True)
(result/'Results').mkdir(exist_ok=True)
(result/"Src/predict_realwaste.py").write_text('"""Prediction using the new TACO checkpoint; keeps the earlier predictor unchanged."""\nimport argparse\nfrom pathlib import Path\nimport torch\nfrom PIL import Image\nfrom torchvision import models, transforms\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'image\')\n    parser.add_argument(\'--model\',default=str(Path(__file__).resolve().parent.parent/\'Models\'/\'ecosort_resnet18_realwaste.pth\'))\n    args=parser.parse_args()\n    checkpoint=torch.load(args.model,map_location=\'cpu\',weights_only=True)\n    classes=checkpoint[\'class_names\']\n    model=models.resnet18(weights=None)\n    model.fc=torch.nn.Linear(model.fc.in_features,len(classes))\n    model.load_state_dict(checkpoint[\'model_state_dict\'])\n    model.eval()\n    transform=transforms.Compose([transforms.Resize((checkpoint[\'image_size\'],checkpoint[\'image_size\'])),\n        transforms.ToTensor(),transforms.Normalize(checkpoint[\'mean\'],checkpoint[\'std\'])])\n    with Image.open(args.image) as image:\n        x=transform(image.convert(\'RGB\')).unsqueeze(0)\n    with torch.inference_mode(): scores=model(x).softmax(1)[0]\n    for i in scores.argsort(descending=True).tolist(): print(f\'{classes[i]}: {scores[i]:.1%}\')\n    print(\'Scores are model outputs, not calibrated probabilities of correctness.\')\n\n\nif __name__==\'__main__\': main()\n')
(result/"README.md").write_text("# Next EcoSort experiment: add RealWaste\n\n## Why add data first\n\nThe present ResNet18 classified four downloaded packaging photos as plastic with top scores of roughly 49–61%. That small positive-only check cannot establish general accuracy, plastic precision or calibrated confidence. Increasing the number of epochs alone may fit the same data more closely without improving performance on new photos.\n\nThis experiment keeps ResNet18 and adds examples from all six compatible RealWaste categories. It compares the previous TACO checkpoint and the new model on exactly the same source-specific test sets after validation-based selection. The previous model remains available. Improvement is an experimental outcome, not a guarantee.\n\n## Data and labels\n\nSource: https://archive.ics.uci.edu/dataset/908/realwaste\n\nCitation: Single, S., Iranmanesh, S., & Raad, R. (2023). RealWaste [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5SS4G. License: CC BY 4.0.\n\n| RealWaste category | EcoSort class | Source images |\n|---|---|---:|\n| Cardboard | cardboard | 461 |\n| Glass | glass | 420 |\n| Metal | metal | 790 |\n| Paper | paper | 500 |\n| Plastic | plastic | 921 |\n| Miscellaneous Trash | trash | 495 |\n| Total used | | 3,587 |\n\nFood Organics, Textile Trash and Vegetation are excluded because the current six-class taxonomy has no explicit matching classes. They are not silently merged into trash. Existing packaging labels remain the project convention: TACO film/wrappers/crisp packets map to plastic, including metallized snack packets. These are not verified composition or recycling instructions. RealWaste's miscellaneous trash category is broad and should be audited against the project's intended meaning of trash.\n\n## Split and actual preparation check\n\nExisting TrashNet and TACO splits are preserved. New RealWaste splitting uses seed 2026 and groups each category's consecutive filename numbers into blocks of 20, then merges near-identical images with whole-image dHash distance <= 4. Groups stay together and are stratified by majority class before approximate 70/15/15 assignment. Exact copies with conflicting labels are excluded; groups containing exact pixel matches to any old source are excluded, and same-class exact copies are reduced to one.\n\nThis is a conservative filename-order heuristic, not verified capture-session grouping. Neither it nor perceptual hashes can detect every related scene. Cross-dataset screening uses exact decoded-pixel hashes, so transformed overlaps may remain. RealWaste represents one landfill environment, not all deployment settings. Source-specific and independent fresh-photo results remain necessary.\n\nLocal preparation on 24 October 2026 checked the downloaded official archive and produced:\n\n| Source | Training | Validation | Test |\n|---|---:|---:|---:|\n| Clean TrashNet | 1,762 | 377 | 382 |\n| TACO packaging | 218 | 45 | 46 |\n| New RealWaste | 2,490 | 575 | 522 |\n| Combined | 4,470 | 997 | 950 |\n\nThese are actual preparation counts, not newly trained GPU results. See `Results/RealWaste_Preparation_Report.json`. Later downloads may vary if upstream files change. No dataset images or model weights are committed to GitHub.\n\n## Current-model validation check\n\nThe actual previously uploaded TACO checkpoint was evaluated on the new RealWaste validation split only: **245/575 correct (42.61% accuracy), macro F1 0.4091**. This does not imply that a newly trained model will improve. The 522 new RealWaste test images have not been evaluated here. Full per-class validation results and checkpoint identity are recorded in `Results/RealWaste_Current_Model_Validation.json`.\n\n## How to run\n\nOpen `NoteBook/EcoSort_TrashNet_TACO_RealWaste.ipynb` in Google Colab with a T4 GPU and run its single code cell.\n\nThe notebook tries to reuse the TrashNet ZIP and current TACO checkpoint in the existing Colab session. If either is missing, it asks for the required files. Keep these two ZIPs available and select the missing ones together in the upload dialog:\n\n- `EcoSort_TrashNet_split.zip`\n- `EcoSort_TACO_Plastic_Result.zip` (the previous trained result bundle)\n\nThe notebook reuses existing TACO crops where available and automatically downloads RealWaste (about 657 MB). It prints the counts and shows training-only examples. It trains a **fresh ImageNet-initialized ResNet18** for up to 15 epochs (3 head, up to 12 layer4/head), with early stopping after 5 epochs without validation selection improvement during fine-tuning. Frozen layers and their batch-normalization statistics stay frozen. Class-weighted loss and augmentation are retained. The selected checkpoint is the highest combined validation macro F1, accuracy breaking ties.\n\nAfter checkpoint selection, the previous and new models are both evaluated on the same test images, separately for TrashNet, TACO and RealWaste. A wrapper-only result is recall, not six-class accuracy or plastic precision. The previously inspected TrashNet and TACO holdouts are diagnostic. Do not use the four online example photos as a new untouched benchmark or add test/validation photos to training.\n\nThe result ZIP is `EcoSort_RealWaste_Result.zip`, with new weights `Models/ecosort_resnet18_realwaste.pth`, metrics/baseline comparison, training history, manifests, class counts, source attribution and package versions. The previous model is not overwritten. The compatible CLI is `Src/predict_realwaste.py`.\n\n## Team tasks\n\n- Prateek: run the new experiment, preserve both result bundles, and review the source-specific comparison before choosing the deployed model. Explain the data/split/model-selection process in the presentation.\n- Prince: review training-only RealWaste categories, especially miscellaneous trash, and build a separate labelled local-photo evaluation set covering all six classes and difficult packaging. Freeze labels before evaluation.\n- Aditya: keep the current app model until the new checkpoint is verified. Make the model path/version configurable; load checkpoint class order and preprocessing. The new filename differs; inference does not require retraining.\n- Rahul: evaluate both checkpoints on the identical frozen local-photo set; report per-class recall/F1, wrapper recall and false plastic predictions, plus app/CLI consistency. Preserve test labels and model SHA-256 checksums.\n\nAfter measuring this data experiment, a separate architecture comparison (for example EfficientNet-B0) can use the same frozen data protocol. Do not change the dataset and architecture simultaneously and attribute the result to one change. Any new architecture also needs a matching inference loader; changing a model filename is insufficient.\n")
versions={n:importlib.metadata.version(n) for n in ['torch','torchvision','Pillow','numpy','scikit-learn']}
(result/'package_versions.json').write_text(json.dumps(versions,indent=2))
with baseline.open('rb') as handle: baseline_sha=hashlib.file_digest(handle,'sha256').hexdigest()
(result/'Results/baseline_identity.json').write_text(json.dumps({'filename':baseline.name,'sha256':baseline_sha},indent=2))
try:
    taco=Path('/content/ecosort_taco_work/taco')
    manifest_path=taco/'manifest.json'
    cached=json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    if not cached or not all((taco/r['file']).exists() for r in cached['records']):
        prepare_taco(taco)
    records=prepare_records(archive,taco,result/'Results',data_root=work/'trashnet')
    print('Downloading/preparing the additional RealWaste images.',flush=True)
    rw_archive=download_realwaste(work/'realwaste.zip')
    rw=prepare_realwaste(rw_archive,records,work/'realwaste')
    records+=rw['records']
    validate_splits(records)
    display(DisplayImage(filename=str(work/'realwaste/training_preview.jpg')))
    from collections import Counter
    summary={'classes':['cardboard','glass','metal','paper','plastic','trash'],
        'split_counts':dict(Counter(r['split'] for r in records)),
        'counts_by_source':{source:dict(Counter(r['split'] for r in records if r['source']==source)) for source in ['trashnet','taco','realwaste']},
        'class_counts_by_split':{s:dict(Counter(r['label'] for r in records if r['split']==s)) for s in ['train','val','test']}}
    print('Combined split counts:',summary['split_counts'],flush=True)
    (result/'Results/combined_dataset_summary.json').write_text(json.dumps(summary,indent=2))
    (result/'Results/dataset_records.json').write_text(json.dumps(records,indent=2))
    for name in ['manifest.json','download_errors.json','training_preview.jpg']:
        shutil.copy2(taco/name,result/'Results'/f'taco_{name}')
    for name in ['manifest.json','training_preview.jpg']:
        shutil.copy2(work/'realwaste'/name,result/'Results'/f'realwaste_{name}')
    train(records,result/'Models',head_epochs=3,finetune_epochs=12,patience=5,
        checkpoint_name='ecosort_resnet18_realwaste.pth',baseline_checkpoint=baseline)
except Exception:
    (result/'error.txt').write_text(traceback.format_exc())
    debug=shutil.make_archive('/content/EcoSort_RealWaste_Debug','zip',result)
    files.download(debug)
    raise
print('Finished. Compare source_test and baseline_source_test in Models/metrics.json.',flush=True)
bundle=shutil.make_archive('/content/EcoSort_RealWaste_Result','zip',result)
files.download(bundle)
